In [1]:
dataset = [("This movie is a masterpiece!", "Positive"),
    ("Not worth watching!", "Negative"),
    ("Terrific!", "Positive")
]

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_length = 128 
formatted_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset]
texts = [text for text, label in formatted_data]
tokenized_data = tokenizer(texts,
               padding=True,
               truncation=True,
               max_length=max_length,
               return_tensors='pt')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/Users/bahloulia/Downloads/agentic_software/Small Language Models/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
import torch
from sklearn.preprocessing import LabelEncoder

input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder().fit_transform([label for _, label in dataset]))

In [6]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = train_test_split(
input_ids, labels, attention_mask, 
random_state=42, test_size=0.1
)

In [7]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask,labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask': 
                 self.attention_mask[idx],
                'labels': self.labels[idx]}

In [8]:
from torch.utils.data import DataLoader

batch_size = 4
train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
formatted_data = [(f"[CLS] {context} [SEP]
                 ➥{target} [SEP]",) for context,
                 ➥ target in dataset]
numerical_data = [tokenizer.encode(example[0],
                ➥ add_special_tokens=True) 
                ➥for example in formatted_data]